In [1]:
import pandas as pd

# 1. Load the datasets
orders = pd.read_csv('orders.csv')
order_details = pd.read_csv('order_details.csv')
pizzas = pd.read_csv('pizzas.csv')
pizza_types = pd.read_csv('pizza_types.csv', encoding='Unicode_escape')

In [3]:
# 2. Basic Cleaning: Convert Date and Time
# Use dayfirst=True to handle dd/mm/yyyy format, or infer_datetime_format for flexibility
orders['date'] = pd.to_datetime(orders['date'], dayfirst=True)
# Keep the time conversion as is - it's working correctly
orders['time'] = pd.to_datetime(orders['time'], format='%H:%M:%S').dt.time

In [7]:
# 3. Ingredient Extraction (The "Explosion")
# Split the ingredients string into a list
pizza_types['ingredient_list'] = pizza_types['ingredients'].str.split(', ')

# Create a separate dataframe for ingredients analysis
ingredients_expanded = pizza_types.explode('ingredient_list')
ingredients_expanded = ingredients_expanded[['pizza_type_id', 'name', 'ingredient_list']]

In [8]:
# 4. Merging Data for Analysis
# We join all tables to calculate which ingredients are used the most based on actual sales
merged_df = order_details.merge(pizzas, on='pizza_id') \
                         .merge(pizza_types, on='pizza_type_id')

In [10]:
# 5. Inventory Optimization: Which ingredients are ordered most?
# Calculate total quantity sold per ingredient
ingredient_sales = merged_df.assign(ingredients=merged_df['ingredients'].str.split(', ')).explode('ingredients')
top_ingredients = ingredient_sales.groupby('ingredients')['quantity'].sum().sort_values(ascending=False)

In [11]:
# 6. Save Cleaned Data for Power BI/SQL
# This creates a 'Master' file that is ready for dashboarding
merged_df.to_csv('cleaned_pizza_sales.csv', index=False)
top_ingredients.to_csv('ingredient_demand.csv')

print("--- Data Cleaning Complete ---")
print(f"Top 5 Ingredients by Demand:\n{top_ingredients.head(5)}")

--- Data Cleaning Complete ---
Top 5 Ingredients by Demand:
ingredients
Garlic               27913
Tomatoes             27052
Red Onions           19834
Red Peppers          16562
Mozzarella Cheese    10569
Name: quantity, dtype: int64
